In [44]:
import os
import optuna
import random
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error
from sklearn.feature_selection import VarianceThreshold


from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor




def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

In [45]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
print(f"Исходные данные загружены. Train: {train.shape}, Test: {test.shape}")


Исходные данные загружены. Train: (751, 214), Test: (250, 211)


In [46]:
target_cols = ['IC50, mM', 'CC50, mM', 'SI']
feature_cols = [col for col in train.columns if col not in ["index"] + target_cols]


In [47]:
medians = train.groupby(feature_cols)[target_cols].transform('median')
train[target_cols] = medians
train_cleaned = train.drop_duplicates(subset=feature_cols, keep='first').reset_index(drop=True)
print(f"После объединения дубликатов молекул по медиане: {train_cleaned.shape}")


После объединения дубликатов молекул по медиане: (630, 214)


In [48]:
train_cleaned = train_cleaned.dropna(subset=target_cols).reset_index(drop=True)
print(f"После удаления строк с пустыми таргетами (NaN): {train_cleaned.shape}")


После удаления строк с пустыми таргетами (NaN): (628, 214)


In [49]:
selector = VarianceThreshold(threshold=0.0)
selector.fit(train_cleaned[feature_cols])
constant_features = [col for col, keep in zip(feature_cols, selector.get_support()) if not keep]
feature_cols = [col for col in feature_cols if col not in constant_features]
print(f"Удалено константных признаков: {len(constant_features)}. Осталось признаков: {len(feature_cols)}")


Удалено константных признаков: 18. Осталось признаков: 192


In [50]:
corr_matrix = train_cleaned[feature_cols].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_features = [column for column in upper_tri.columns if any(upper_tri[column] > 0.95)]
feature_cols = [col for col in feature_cols if col not in high_corr_features]
print(f"Удалено сильно коррелирующих признаков (>0.95): {len(high_corr_features)}. Итого признаков: {len(feature_cols)}")


Удалено сильно коррелирующих признаков (>0.95): 34. Итого признаков: 158


In [51]:
q25 = train_cleaned['SI'].quantile(0.25)
q75 = train_cleaned['SI'].quantile(0.75)
iqr = q75 - q25
upper_boundary = q75 + 3.0 * iqr
train_final = train_cleaned[train_cleaned['SI'] <= upper_boundary].reset_index(drop=True)
print(f"Граница выбросов по IQR для SI: {upper_boundary:.2f}. Удалено выбросов: {train_cleaned.shape[0] - train_final.shape[0]}")


Граница выбросов по IQR для SI: 48.59. Удалено выбросов: 49


In [52]:
train_final['fold'] = -1
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (train_idx, val_idx) in enumerate(kf.split(train_final)):
    train_final.loc[val_idx, 'fold'] = fold_idx

print(f"\n Очищенный датасет")
print(f"Размерность train_final: {train_final.shape}")
print(f"Количество признаков в feature_cols: {len(feature_cols)}")
print(f"Распределение строк по 5 фолдам:\n{train_final['fold'].value_counts().to_string()}")


 Очищенный датасет
Размерность train_final: (579, 215)
Количество признаков в feature_cols: 158
Распределение строк по 5 фолдам:
fold
1    116
0    116
2    116
3    116
4    115


In [53]:
train_final = train_final.copy()


In [54]:
models_ic50 = []
models_cc50 = []
models_si = []

train_final['oof_IC50'] = 0.0
train_final['oof_CC50'] = 0.0

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
    model.fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'],
              eval_set=(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM']),
              early_stopping_rounds=100, verbose=False)

    train_final.loc[val_idx, 'oof_IC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    models_ic50.append(model)

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
    model.fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'],
              eval_set=(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM']),
              early_stopping_rounds=100, verbose=False)

    train_final.loc[val_idx, 'oof_CC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    models_cc50.append(model)

feature_cols_si = feature_cols + ['oof_IC50', 'oof_CC50']
for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
    model.fit(train_final.loc[train_idx, feature_cols_si], train_final.loc[train_idx, 'SI'],
              eval_set=(train_final.loc[val_idx, feature_cols_si], train_final.loc[val_idx, 'SI']),
              early_stopping_rounds=100, verbose=False)
    models_si.append(model)

test_preds_ic50 = np.zeros(len(test))
test_preds_cc50 = np.zeros(len(test))
test_preds_si = np.zeros(len(test))

for model in models_ic50:
    test_preds_ic50 += model.predict(test[feature_cols]) / 5

for model in models_cc50:
    test_preds_cc50 += model.predict(test[feature_cols]) / 5

test_meta = test.copy()
test_meta['oof_IC50'] = test_preds_ic50
test_meta['oof_CC50'] = test_preds_cc50

for model in models_si:
    test_preds_si += model.predict(test_meta[feature_cols_si]) / 5


submission = pd.DataFrame({
    'index': test['index'],
    'IC50': test_preds_ic50,
    'CC50': test_preds_cc50,
    'SI': test_preds_si
})

submission.to_csv('submission.csv', index=False)
print("Файл submission.csv создан. Формат:")
print(submission.head())
print(f"\nРазмерность файла: {submission.shape}")

Файл submission.csv создан. Формат:
   index        IC50        CC50        SI
0      0  173.109074  360.244931  8.202102
1      1  235.091709  379.337866  5.609795
2      2  158.658089  305.884217  8.568503
3      3  281.977935  396.177885  6.447149
4      4  207.574996  357.836030  4.830452

Размерность файла: (250, 4)


In [55]:
xgb_models_ic50 = []
xgb_models_cc50 = []
xgb_models_si = []

train_final['xgb_oof_IC50'] = 0.0
train_final['xgb_oof_CC50'] = 0.0

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100)
    model.fit(
        train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'],
        eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM'])],
        verbose=False
    )

    train_final.loc[val_idx, 'xgb_oof_IC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    xgb_models_ic50.append(model)

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100)
    model.fit(
        train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'],
        eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM'])],
        verbose=False
    )

    train_final.loc[val_idx, 'xgb_oof_CC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    xgb_models_cc50.append(model)

feature_cols_xgb_si = feature_cols + ['xgb_oof_IC50', 'xgb_oof_CC50']
xgb_oof_predictions_si = np.zeros(len(train_final))

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100)
    model.fit(
        train_final.loc[train_idx, feature_cols_xgb_si], train_final.loc[train_idx, 'SI'],
        eval_set=[(train_final.loc[val_idx, feature_cols_xgb_si], train_final.loc[val_idx, 'SI'])],
        verbose=False
    )
    xgb_oof_predictions_si[val_idx] = model.predict(train_final.loc[val_idx, feature_cols_xgb_si])
    xgb_models_si.append(model)

xgb_test_ic50 = np.zeros(len(test))
xgb_test_cc50 = np.zeros(len(test))
xgb_test_si = np.zeros(len(test))

for model in xgb_models_ic50:
    xgb_test_ic50 += model.predict(test[feature_cols]) / 5

for model in xgb_models_cc50:
    xgb_test_cc50 += model.predict(test[feature_cols]) / 5

test_meta_xgb = test.copy()
test_meta_xgb['xgb_oof_IC50'] = xgb_test_ic50
test_meta_xgb['xgb_oof_CC50'] = xgb_test_cc50

for model in xgb_models_si:
    xgb_test_si += model.predict(test_meta_xgb[feature_cols_xgb_si]) / 5

print("Обучение XGBoost завершено")
print(f"Локальный OOF RMSE для IC50 (XGB): {root_mean_squared_error(train_final['IC50, mM'], train_final['xgb_oof_IC50']):.4f}")
print(f"Локальный OOF RMSE для CC50 (XGB): {root_mean_squared_error(train_final['CC50, mM'], train_final['xgb_oof_CC50']):.4f}")
print(f"Локальный OOF RMSE для SI (XGB): {root_mean_squared_error(train_final['SI'], xgb_oof_predictions_si):.4f}")


Обучение XGBoost завершено
Локальный OOF RMSE для IC50 (XGB): 330.7210
Локальный OOF RMSE для CC50 (XGB): 451.3674
Локальный OOF RMSE для SI (XGB): 9.5953


In [56]:
blended_ic50 = (test_preds_ic50 + xgb_test_ic50) / 2
blended_cc50 = (test_preds_cc50 + xgb_test_cc50) / 2
blended_si = (test_preds_si + xgb_test_si) / 2
submission_blended = pd.DataFrame({
    'index': test['index'],
    'IC50': blended_ic50,
    'CC50': blended_cc50,
    'SI': blended_si
})

submission_blended.to_csv('submission_blended.csv', index=False)

print("Файл submission_blended.csv создан")
print(submission_blended.head())

Файл submission_blended.csv создан
   index        IC50        CC50        SI
0      0  183.376287  407.301821  8.574248
1      1  237.705301  390.244605  5.849062
2      2  153.684235  346.939434  8.095828
3      3  303.942306  438.748698  6.133202
4      4  198.019498  353.846655  5.372371


In [57]:
lgb_models_ic50 = []
lgb_models_cc50 = []
lgb_models_si = []

train_final['lgb_oof_IC50'] = 0.0
train_final['lgb_oof_CC50'] = 0.0

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(
        train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'],
        eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM'])],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )

    train_final.loc[val_idx, 'lgb_oof_IC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    lgb_models_ic50.append(model)

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(
        train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'],
        eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM'])],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )

    train_final.loc[val_idx, 'lgb_oof_CC50'] = model.predict(train_final.loc[val_idx, feature_cols])
    lgb_models_cc50.append(model)

feature_cols_lgb_si = feature_cols + ['lgb_oof_IC50', 'lgb_oof_CC50']
lgb_oof_predictions_si = np.zeros(len(train_final))

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    model = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(
        train_final.loc[train_idx, feature_cols_lgb_si], train_final.loc[train_idx, 'SI'],
        eval_set=[(train_final.loc[val_idx, feature_cols_lgb_si], train_final.loc[val_idx, 'SI'])],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    lgb_oof_predictions_si[val_idx] = model.predict(train_final.loc[val_idx, feature_cols_lgb_si])
    lgb_models_si.append(model)

lgb_test_ic50 = np.zeros(len(test))
lgb_test_cc50 = np.zeros(len(test))
lgb_test_si = np.zeros(len(test))

for model in lgb_models_ic50:
    lgb_test_ic50 += model.predict(test[feature_cols]) / 5

for model in lgb_models_cc50:
    lgb_test_cc50 += model.predict(test[feature_cols]) / 5

test_meta_lgb = test.copy()
test_meta_lgb['lgb_oof_IC50'] = lgb_test_ic50
test_meta_lgb['lgb_oof_CC50'] = lgb_test_cc50

for model in lgb_models_si:
    lgb_test_si += model.predict(test_meta_lgb[feature_cols_lgb_si]) / 5

print("Обучение LightGBM завершено")
print(f"Локальный OOF RMSE для IC50 (LGB): {root_mean_squared_error(train_final['IC50, mM'], train_final['lgb_oof_IC50']):.4f}")
print(f"Локальный OOF RMSE для CC50 (LGB): {root_mean_squared_error(train_final['CC50, mM'], train_final['lgb_oof_CC50']):.4f}")
print(f"Локальный OOF RMSE для SI (LGB): {root_mean_squared_error(train_final['SI'], lgb_oof_predictions_si):.4f}")


Обучение LightGBM завершено
Локальный OOF RMSE для IC50 (LGB): 328.7775
Локальный OOF RMSE для CC50 (LGB): 456.3897
Локальный OOF RMSE для SI (LGB): 9.5069


In [58]:
triple_blended_ic50 = (test_preds_ic50 + xgb_test_ic50 + lgb_test_ic50) / 3
triple_blended_cc50 = (test_preds_cc50 + xgb_test_cc50 + lgb_test_cc50) / 3
triple_blended_si = (test_preds_si + xgb_test_si + lgb_test_si) / 3

submission_triple = pd.DataFrame({
    'index': test['index'],
    'IC50': triple_blended_ic50,
    'CC50': triple_blended_cc50,
    'SI': triple_blended_si
})

submission_triple.to_csv('submission_triple_blend.csv', index=False)

print("Файл submission_triple_blend.csv создан")
print(submission_triple.head())


Файл submission_triple_blend.csv создан
   index        IC50        CC50        SI
0      0  191.264482  397.777092  8.415824
1      1  241.069216  398.769699  5.985821
2      2  157.073158  412.768885  8.002162
3      3  306.050083  410.405055  5.950809
4      4  209.663371  326.389420  5.233820


In [59]:
warnings.filterwarnings('ignore')
def objective_catboost_ic50(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 1200),
        'depth': trial.suggest_int('depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'random_seed': 42,
        'verbose': 0
    }

    fold_errors = []

    for fold in range(5):
        train_idx = train_final[train_final['fold'] != fold].index
        val_idx = train_final[train_final['fold'] == fold].index

        X_train = train_final.loc[train_idx, feature_cols]
        y_train = train_final.loc[train_idx, 'IC50, mM']
        X_val = train_final.loc[val_idx, feature_cols]
        y_val = train_final.loc[val_idx, 'IC50, mM']

        model = CatBoostRegressor(**params)
        model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50, verbose=False)

        preds = model.predict(X_val)
        fold_rmse = root_mean_squared_error(y_val, preds)
        fold_errors.append(fold_rmse)


    return np.mean(fold_errors)

study = optuna.create_study(direction='minimize')

print(" тюнинг гиперпараметров через Optuna")
study.optimize(objective_catboost_ic50, n_trials=10)

print("\n завершено")
print(f"Лучший локальный RMSE для IC50: {study.best_value:.4f}")
print("Идеальные параметры:", study.best_params)


[I 2026-05-25 17:37:01,397] A new study created in memory with name: no-name-8835f144-963a-468d-b342-dabba083b9a7


 тюнинг гиперпараметров через Optuna


[I 2026-05-25 17:37:07,769] Trial 0 finished with value: 324.9792046959577 and parameters: {'iterations': 521, 'depth': 5, 'learning_rate': 0.029042254660847547, 'l2_leaf_reg': 3.6354331794858226}. Best is trial 0 with value: 324.9792046959577.
[I 2026-05-25 17:37:27,296] Trial 1 finished with value: 330.21079109173735 and parameters: {'iterations': 967, 'depth': 8, 'learning_rate': 0.08664773008777818, 'l2_leaf_reg': 8.165422273590009}. Best is trial 0 with value: 324.9792046959577.
[I 2026-05-25 17:37:44,399] Trial 2 finished with value: 327.4346125242223 and parameters: {'iterations': 1115, 'depth': 8, 'learning_rate': 0.042158406937019235, 'l2_leaf_reg': 1.9586285623731057}. Best is trial 0 with value: 324.9792046959577.
[I 2026-05-25 17:38:20,821] Trial 3 finished with value: 328.41465221509424 and parameters: {'iterations': 1168, 'depth': 8, 'learning_rate': 0.041094455023600214, 'l2_leaf_reg': 7.885812226706568}. Best is trial 0 with value: 324.9792046959577.
[I 2026-05-25 17:38


 завершено
Лучший локальный RMSE для IC50: 324.9792
Идеальные параметры: {'iterations': 521, 'depth': 5, 'learning_rate': 0.029042254660847547, 'l2_leaf_reg': 3.6354331794858226}


In [60]:

def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1200),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0), # L2 регуляризация
        'random_state': 42,
        'n_jobs': -1
    }
    errors = []
    for fold in range(5):
        train_idx, val_idx = train_final[train_final['fold'] != fold].index, train_final[train_final['fold'] == fold].index
        model = XGBRegressor(**params, early_stopping_rounds=50)
        model.fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'],
                  eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM'])], verbose=False)
        errors.append(root_mean_squared_error(train_final.loc[val_idx, 'IC50, mM'], model.predict(train_final.loc[val_idx, feature_cols])))
    return np.mean(errors)

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=10)
print(f"Лучший RMSE для XGBoost: {study_xgb.best_value:.4f}")

def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1200),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    errors = []
    for fold in range(5):
        train_idx, val_idx = train_final[train_final['fold'] != fold].index, train_final[train_final['fold'] == fold].index
        model = LGBMRegressor(**params)
        model.fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'],
                  eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM'])],
                  callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
        errors.append(root_mean_squared_error(train_final.loc[val_idx, 'IC50, mM'], model.predict(train_final.loc[val_idx, feature_cols])))
    return np.mean(errors)

study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=10)
print(f"Лучший RMSE для LightGBM: {study_lgb.best_value:.4f}")


[I 2026-05-25 17:40:00,837] A new study created in memory with name: no-name-2933284e-ad8e-4ed2-8911-e811cea21514
[I 2026-05-25 17:40:03,104] Trial 0 finished with value: 323.19797998541253 and parameters: {'n_estimators': 874, 'max_depth': 4, 'learning_rate': 0.04228199008723936, 'reg_lambda': 8.06227397975187}. Best is trial 0 with value: 323.19797998541253.
[I 2026-05-25 17:40:05,910] Trial 1 finished with value: 327.52100015133783 and parameters: {'n_estimators': 1082, 'max_depth': 6, 'learning_rate': 0.0424065092372032, 'reg_lambda': 5.9321958032058255}. Best is trial 0 with value: 323.19797998541253.
[I 2026-05-25 17:40:08,688] Trial 2 finished with value: 323.44634567404125 and parameters: {'n_estimators': 1074, 'max_depth': 4, 'learning_rate': 0.02576208621213239, 'reg_lambda': 9.279894612684648}. Best is trial 0 with value: 323.19797998541253.
[I 2026-05-25 17:40:12,705] Trial 3 finished with value: 325.8596178033503 and parameters: {'n_estimators': 955, 'max_depth': 7, 'learn

Лучший RMSE для XGBoost: 321.1071


[I 2026-05-25 17:40:37,115] Trial 0 finished with value: 324.3003318106957 and parameters: {'n_estimators': 1061, 'max_depth': 4, 'learning_rate': 0.028869090646712035, 'reg_lambda': 5.671984537042526}. Best is trial 0 with value: 324.3003318106957.
[I 2026-05-25 17:40:37,451] Trial 1 finished with value: 323.33667600801124 and parameters: {'n_estimators': 884, 'max_depth': 4, 'learning_rate': 0.07801347608368348, 'reg_lambda': 3.969375805802715}. Best is trial 1 with value: 323.33667600801124.
[I 2026-05-25 17:40:37,931] Trial 2 finished with value: 322.49868359800337 and parameters: {'n_estimators': 1125, 'max_depth': 5, 'learning_rate': 0.07146211674589009, 'reg_lambda': 5.322867343802248}. Best is trial 2 with value: 322.49868359800337.
[I 2026-05-25 17:40:38,826] Trial 3 finished with value: 321.98524004607555 and parameters: {'n_estimators': 1189, 'max_depth': 6, 'learning_rate': 0.021963007045080765, 'reg_lambda': 9.173329030765865}. Best is trial 3 with value: 321.9852400460755

Лучший RMSE для LightGBM: 321.9852


In [61]:
cb_params = {'iterations': 740, 'depth': 6, 'learning_rate': 0.0796, 'l2_leaf_reg': 5.52, 'random_seed': 42, 'verbose': 0}
xgb_params = {'n_estimators': 655, 'max_depth': 4, 'learning_rate': 0.0855, 'reg_lambda': 7.94, 'random_state': 42, 'n_jobs': -1}
lgb_params = {'n_estimators': 875, 'max_depth': 6, 'learning_rate': 0.0449, 'reg_lambda': 2.22, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}

opt_test_ic50_cb, opt_test_ic50_xgb, opt_test_ic50_lgb = np.zeros(len(test)), np.zeros(len(test)), np.zeros(len(test))
opt_test_cc50_cb, opt_test_cc50_xgb, opt_test_cc50_lgb = np.zeros(len(test)), np.zeros(len(test)), np.zeros(len(test))
opt_test_si_cb, opt_test_si_xgb, opt_test_si_lgb = np.zeros(len(test)), np.zeros(len(test)), np.zeros(len(test))

train_final['opt_oof_IC50'] = 0.0
train_final['opt_xgb_oof_IC50'] = 0.0
train_final['opt_lgb_oof_IC50'] = 0.0

train_final['opt_oof_CC50'] = 0.0
train_final['opt_xgb_oof_CC50'] = 0.0
train_final['opt_lgb_oof_CC50'] = 0.0

for fold in range(5):
    train_idx, val_idx = train_final[train_final['fold'] != fold].index, train_final[train_final['fold'] == fold].index

    # CatBoost
    m_cb = CatBoostRegressor(**cb_params).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'], eval_set=(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM']), early_stopping_rounds=50, verbose=False)
    train_final.loc[val_idx, 'opt_oof_IC50'] = m_cb.predict(train_final.loc[val_idx, feature_cols])
    opt_test_ic50_cb += m_cb.predict(test[feature_cols]) / 5

    # XGBoost
    m_xgb = XGBRegressor(**xgb_params, early_stopping_rounds=50).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'], eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM'])], verbose=False)
    train_final.loc[val_idx, 'opt_xgb_oof_IC50'] = m_xgb.predict(train_final.loc[val_idx, feature_cols])
    opt_test_ic50_xgb += m_xgb.predict(test[feature_cols]) / 5

    # LightGBM
    m_lgb = LGBMRegressor(**lgb_params).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'], eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM'])], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
    train_final.loc[val_idx, 'opt_lgb_oof_IC50'] = m_lgb.predict(train_final.loc[val_idx, feature_cols])
    opt_test_ic50_lgb += m_lgb.predict(test[feature_cols]) / 5

for fold in range(5):
    train_idx, val_idx = train_final[train_final['fold'] != fold].index, train_final[train_final['fold'] == fold].index

    m_cb = CatBoostRegressor(**cb_params).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'], eval_set=(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM']), early_stopping_rounds=50, verbose=False)
    train_final.loc[val_idx, 'opt_oof_CC50'] = m_cb.predict(train_final.loc[val_idx, feature_cols])
    opt_test_cc50_cb += m_cb.predict(test[feature_cols]) / 5

    m_xgb = XGBRegressor(**xgb_params, early_stopping_rounds=50).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'], eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM'])], verbose=False)
    train_final.loc[val_idx, 'opt_xgb_oof_CC50'] = m_xgb.predict(train_final.loc[val_idx, feature_cols])
    opt_test_cc50_xgb += m_xgb.predict(test[feature_cols]) / 5

    m_lgb = LGBMRegressor(**lgb_params).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'], eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM'])], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
    train_final.loc[val_idx, 'opt_lgb_oof_CC50'] = m_lgb.predict(train_final.loc[val_idx, feature_cols])
    opt_test_cc50_lgb += m_lgb.predict(test[feature_cols]) / 5

for fold in range(5):
    train_idx, val_idx = train_final[train_final['fold'] != fold].index, train_final[train_final['fold'] == fold].index

    f_cb = feature_cols + ['opt_oof_IC50', 'opt_oof_CC50']
    f_xgb = feature_cols + ['opt_xgb_oof_IC50', 'opt_xgb_oof_CC50']
    f_lgb = feature_cols + ['opt_lgb_oof_IC50', 'opt_lgb_oof_CC50']

    t_cb, t_xgb, t_lgb = test.copy(), test.copy(), test.copy()
    t_cb['opt_oof_IC50'], t_cb['opt_oof_CC50'] = opt_test_ic50_cb, opt_test_cc50_cb
    t_xgb['opt_xgb_oof_IC50'], t_xgb['opt_xgb_oof_CC50'] = opt_test_ic50_xgb, opt_test_cc50_xgb
    t_lgb['opt_lgb_oof_IC50'], t_lgb['opt_lgb_oof_CC50'] = opt_test_ic50_lgb, opt_test_cc50_lgb


    m_cb = CatBoostRegressor(**cb_params).fit(train_final.loc[train_idx, f_cb], train_final.loc[train_idx, 'SI'], eval_set=(train_final.loc[val_idx, f_cb], train_final.loc[val_idx, 'SI']), early_stopping_rounds=50, verbose=False)
    opt_test_si_cb += m_cb.predict(t_cb[f_cb]) / 5

    m_xgb = XGBRegressor(**xgb_params, early_stopping_rounds=50).fit(train_final.loc[train_idx, f_xgb], train_final.loc[train_idx, 'SI'], eval_set=[(train_final.loc[val_idx, f_xgb], train_final.loc[val_idx, 'SI'])], verbose=False)
    opt_test_si_xgb += m_xgb.predict(t_xgb[f_xgb]) / 5

    m_lgb = LGBMRegressor(**lgb_params).fit(train_final.loc[train_idx, f_lgb], train_final.loc[train_idx, 'SI'], eval_set=[(train_final.loc[val_idx, f_lgb], train_final.loc[val_idx, 'SI'])], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
    opt_test_si_lgb += m_lgb.predict(t_lgb[f_lgb]) / 5

final_ic50 = (opt_test_ic50_cb + opt_test_ic50_xgb + opt_test_ic50_lgb) / 3
final_cc50 = (opt_test_cc50_cb + opt_test_cc50_xgb + opt_test_cc50_lgb) / 3
final_si   = (opt_test_si_cb + opt_test_si_xgb + opt_test_si_lgb) / 3

submission_optuna = pd.DataFrame({
    'index': test['index'],
    'IC50': final_ic50,
    'CC50': final_cc50,
    'SI': final_si
})

submission_optuna.to_csv('submission_optuna_blend.csv', index=False)
print("файл submission_optuna_blend.csv создан")
print(submission_optuna.head())


файл submission_optuna_blend.csv создан
   index        IC50        CC50        SI
0      0  192.747669  402.152946  7.944375
1      1  245.899287  408.238617  5.487991
2      2  139.849060  428.309071  7.719802
3      3  270.212158  408.904417  5.910707
4      4  218.027337  329.315638  4.794091


In [62]:
test_clean = test.copy()
test_clean[feature_cols] = test_clean[feature_cols].replace([np.inf, -np.inf], np.nan)

train_final_clean = train_final.copy()
train_final_clean[feature_cols] = train_final_clean[feature_cols].replace([np.inf, -np.inf], np.nan)

train_final_clean['ridge_oof_IC50'] = 0.0
train_final_clean['ridge_oof_CC50'] = 0.0
ridge_oof_predictions_si = np.zeros(len(train_final_clean))

ridge_models_ic50, ridge_scalers_ic50, ridge_imputers_ic50 = [], [], []
ridge_models_cc50, ridge_scalers_cc50, ridge_imputers_cc50 = [], [], []
ridge_models_si, ridge_scalers_si, ridge_imputers_si = [], [], []

for fold in range(5):
    train_idx = train_final_clean[train_final_clean['fold'] != fold].index
    val_idx = train_final_clean[train_final_clean['fold'] == fold].index

    imputer = SimpleImputer(strategy='median')
    scaler = StandardScaler()

    X_train_imp = imputer.fit_transform(train_final_clean.loc[train_idx, feature_cols])
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_val_imp = imputer.transform(train_final_clean.loc[val_idx, feature_cols])
    X_val_scaled = scaler.transform(X_val_imp)

    model = Ridge(alpha=10.0, random_state=42)
    model.fit(X_train_scaled, train_final_clean.loc[train_idx, 'IC50, mM'])

    train_final_clean.loc[val_idx, 'ridge_oof_IC50'] = model.predict(X_val_scaled)
    ridge_models_ic50.append(model)
    ridge_scalers_ic50.append(scaler)
    ridge_imputers_ic50.append(imputer)

for fold in range(5):
    train_idx = train_final_clean[train_final_clean['fold'] != fold].index
    val_idx = train_final_clean[train_final_clean['fold'] == fold].index

    imputer = SimpleImputer(strategy='median')
    scaler = StandardScaler()

    X_train_imp = imputer.fit_transform(train_final_clean.loc[train_idx, feature_cols])
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_val_imp = imputer.transform(train_final_clean.loc[val_idx, feature_cols])
    X_val_scaled = scaler.transform(X_val_imp)

    model = Ridge(alpha=10.0, random_state=42)
    model.fit(X_train_scaled, train_final_clean.loc[train_idx, 'CC50, mM'])

    train_final_clean.loc[val_idx, 'ridge_oof_CC50'] = model.predict(X_val_scaled)
    ridge_models_cc50.append(model)
    ridge_scalers_cc50.append(scaler)
    ridge_imputers_cc50.append(imputer)

feature_cols_ridge_si = feature_cols + ['ridge_oof_IC50', 'ridge_oof_CC50']

for fold in range(5):
    train_idx = train_final_clean[train_final_clean['fold'] != fold].index
    val_idx = train_final_clean[train_final_clean['fold'] == fold].index

    imputer = SimpleImputer(strategy='median')
    scaler = StandardScaler()

    X_train_imp = imputer.fit_transform(train_final_clean.loc[train_idx, feature_cols_ridge_si])
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_val_imp = imputer.transform(train_final_clean.loc[val_idx, feature_cols_ridge_si])
    X_val_scaled = scaler.transform(X_val_imp)

    model = Ridge(alpha=10.0, random_state=42)
    model.fit(X_train_scaled, train_final_clean.loc[train_idx, 'SI'])

    ridge_oof_predictions_si[val_idx] = model.predict(X_val_scaled)
    ridge_models_si.append(model)
    ridge_scalers_si.append(scaler)
    ridge_imputers_si.append(imputer)

ridge_test_ic50 = np.zeros(len(test_clean))
ridge_test_cc50 = np.zeros(len(test_clean))
ridge_test_si = np.zeros(len(test_clean))

for model, scaler, imputer in zip(ridge_models_ic50, ridge_scalers_ic50, ridge_imputers_ic50):
    test_imp = imputer.transform(test_clean[feature_cols])
    ridge_test_ic50 += model.predict(scaler.transform(test_imp)) / 5

for model, scaler, imputer in zip(ridge_models_cc50, ridge_scalers_cc50, ridge_imputers_cc50):
    test_imp = imputer.transform(test_clean[feature_cols])
    ridge_test_cc50 += model.predict(scaler.transform(test_imp)) / 5

test_meta_ridge = test_clean.copy()
test_meta_ridge['ridge_oof_IC50'] = ridge_test_ic50
test_meta_ridge['ridge_oof_CC50'] = ridge_test_cc50

for model, scaler, imputer in zip(ridge_models_si, ridge_scalers_si, ridge_imputers_si):
    test_imp = imputer.transform(test_meta_ridge[feature_cols_ridge_si])
    ridge_test_si += model.predict(scaler.transform(test_imp)) / 5

print("Обучение Ridge-регрессии завершено")
print(f"Локальный OOF RMSE для IC50 (Ridge): {root_mean_squared_error(train_final_clean['IC50, mM'], train_final_clean['ridge_oof_IC50']):.4f}")
print(f"Локальный OOF RMSE для CC50 (Ridge): {root_mean_squared_error(train_final_clean['CC50, mM'], train_final_clean['ridge_oof_CC50']):.4f}")
print(f"Локальный OOF RMSE для SI (Ridge): {root_mean_squared_error(train_final_clean['SI'], ridge_oof_predictions_si):.4f}")


Обучение Ridge-регрессии завершено
Локальный OOF RMSE для IC50 (Ridge): 365.9741
Локальный OOF RMSE для CC50 (Ridge): 523.4762
Локальный OOF RMSE для SI (Ridge): 10.2169


In [63]:
def generate_chemical_features(df):
    df_copy = df.copy()
    eps = 1e-5
    df_copy['custom_MolWt_per_Atom'] = df_copy['MolWt'] / (df_copy['HeavyAtomCount'] + eps)
    df_copy['custom_Polar_Ratio'] = df_copy['TPSA'] / (df_copy['LabuteASA'] + eps)
    df_copy['custom_Hydrophobic_Efficiency'] = df_copy['MolLogP'] / (df_copy['LabuteASA'] + eps)
    df_copy['custom_Electrons_per_Atom'] = df_copy['NumValenceElectrons'] / (df_copy['HeavyAtomCount'] + eps)
    return df_copy

train_final = generate_chemical_features(train_final)
test = generate_chemical_features(test)

new_features = ['custom_MolWt_per_Atom', 'custom_Polar_Ratio', 'custom_Hydrophobic_Efficiency', 'custom_Electrons_per_Atom']
feature_cols = feature_cols + new_features

print(f"Новое количество признаков в feature_cols: {len(feature_cols)}")

Новое количество признаков в feature_cols: 162


In [64]:
feature_cols = list(set(feature_cols))
oof_cols_to_clear = ['oof_IC50', 'oof_CC50', 'xgb_oof_IC50', 'xgb_oof_CC50', 'lgb_oof_IC50', 'lgb_oof_CC50', 'ridge_oof_IC50', 'ridge_oof_CC50']
for col in oof_cols_to_clear:
    train_final[col] = 0.0

test_preds_ic50, xgb_test_ic50, lgb_test_ic50, ridge_test_ic50 = np.zeros(len(test)), np.zeros(len(test)), np.zeros(len(test)), np.zeros(len(test))
test_preds_cc50, xgb_test_cc50, lgb_test_cc50, ridge_test_cc50 = np.zeros(len(test)), np.zeros(len(test)), np.zeros(len(test)), np.zeros(len(test))

for fold in range(5):
    train_idx, val_idx = train_final[train_final['fold'] != fold].index, train_final[train_final['fold'] == fold].index

    m_cb_ic = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'], eval_set=(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM']), early_stopping_rounds=100, verbose=False)
    train_final.loc[val_idx, 'oof_IC50'] = m_cb_ic.predict(train_final.loc[val_idx, feature_cols])
    test_preds_ic50 += m_cb_ic.predict(test[feature_cols]) / 5

    m_cb_cc = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'], eval_set=(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM']), early_stopping_rounds=100, verbose=False)
    train_final.loc[val_idx, 'oof_CC50'] = m_cb_cc.predict(train_final.loc[val_idx, feature_cols])
    test_preds_cc50 += m_cb_cc.predict(test[feature_cols]) / 5

    m_xgb_ic = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'], eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM'])], verbose=False)
    train_final.loc[val_idx, 'xgb_oof_IC50'] = m_xgb_ic.predict(train_final.loc[val_idx, feature_cols])
    xgb_test_ic50 += m_xgb_ic.predict(test[feature_cols]) / 5

    m_xgb_cc = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'], eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM'])], verbose=False)
    train_final.loc[val_idx, 'xgb_oof_CC50'] = m_xgb_cc.predict(train_final.loc[val_idx, feature_cols])
    xgb_test_cc50 += m_xgb_cc.predict(test[feature_cols]) / 5

    m_lgb_ic = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'IC50, mM'], eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'IC50, mM'])], callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
    train_final.loc[val_idx, 'lgb_oof_IC50'] = m_lgb_ic.predict(train_final.loc[val_idx, feature_cols])
    lgb_test_ic50 += m_lgb_ic.predict(test[feature_cols]) / 5

    m_lgb_cc = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1).fit(train_final.loc[train_idx, feature_cols], train_final.loc[train_idx, 'CC50, mM'], eval_set=[(train_final.loc[val_idx, feature_cols], train_final.loc[val_idx, 'CC50, mM'])], callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
    train_final.loc[val_idx, 'lgb_oof_CC50'] = m_lgb_cc.predict(train_final.loc[val_idx, feature_cols])
    lgb_test_cc50 += m_lgb_cc.predict(test[feature_cols]) / 5

    imp, scl = SimpleImputer(strategy='median'), StandardScaler()
    X_tr_scaled = scl.fit_transform(imp.fit_transform(train_final.loc[train_idx, feature_cols]))
    X_va_scaled = scl.transform(imp.transform(train_final.loc[val_idx, feature_cols]))
    t_imp_scaled = scl.transform(imp.transform(test[feature_cols]))

    m_rg_ic = Ridge(alpha=10.0, random_state=42).fit(X_tr_scaled, train_final.loc[train_idx, 'IC50, mM'])
    train_final.loc[val_idx, 'ridge_oof_IC50'] = m_rg_ic.predict(X_va_scaled)
    ridge_test_ic50 += m_rg_ic.predict(t_imp_scaled) / 5

    m_rg_cc = Ridge(alpha=10.0, random_state=42).fit(X_tr_scaled, train_final.loc[train_idx, 'CC50, mM'])
    train_final.loc[val_idx, 'ridge_oof_CC50'] = m_rg_cc.predict(X_va_scaled)
    ridge_test_cc50 += m_rg_cc.predict(t_imp_scaled) / 5

test_preds_si, xgb_test_si, lgb_test_si, ridge_test_si = np.zeros(len(test)), np.zeros(len(test)), np.zeros(len(test)), np.zeros(len(test))

for fold in range(5):
    train_idx, val_idx = train_final[train_final['fold'] != fold].index, train_final[train_final['fold'] == fold].index

    f_cb, f_xgb, f_lgb, f_rg = feature_cols + ['oof_IC50', 'oof_CC50'], feature_cols + ['xgb_oof_IC50', 'xgb_oof_CC50'], feature_cols + ['lgb_oof_IC50', 'lgb_oof_CC50'], feature_cols + ['ridge_oof_IC50', 'ridge_oof_CC50']
    t_cb, t_xgb, t_lgb, t_rg = test.copy(), test.copy(), test.copy(), test.copy()
    t_cb['oof_IC50'], t_cb['oof_CC50'] = test_preds_ic50, test_preds_cc50
    t_xgb['xgb_oof_IC50'], t_xgb['xgb_oof_CC50'] = xgb_test_ic50, xgb_test_cc50
    t_lgb['lgb_oof_IC50'], t_lgb['lgb_oof_CC50'] = lgb_test_ic50, lgb_test_cc50
    t_rg['ridge_oof_IC50'], t_rg['ridge_oof_CC50'] = ridge_test_ic50, ridge_test_cc50

    m_cb = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0).fit(train_final.loc[train_idx, f_cb], train_final.loc[train_idx, 'SI'], eval_set=(train_final.loc[val_idx, f_cb], train_final.loc[val_idx, 'SI']), early_stopping_rounds=100, verbose=False)
    test_preds_si += m_cb.predict(t_cb[f_cb]) / 5

    m_xgb = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100).fit(train_final.loc[train_idx, f_xgb], train_final.loc[train_idx, 'SI'], eval_set=[(train_final.loc[val_idx, f_xgb], train_final.loc[val_idx, 'SI'])], verbose=False)
    xgb_test_si += m_xgb.predict(t_xgb[f_xgb]) / 5

    m_lgb = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1).fit(train_final.loc[train_idx, f_lgb], train_final.loc[train_idx, 'SI'], eval_set=[(train_final.loc[val_idx, f_lgb], train_final.loc[val_idx, 'SI'])], callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
    lgb_test_si += m_lgb.predict(t_lgb[f_lgb]) / 5

    X_tr_sc = scl.fit_transform(imp.fit_transform(train_final.loc[train_idx, f_rg]))
    ridge_test_si += Ridge(alpha=10.0, random_state=42).fit(X_tr_sc, train_final.loc[train_idx, 'SI']).predict(scl.transform(imp.transform(t_rg[f_rg]))) / 5

final_blend_ic50 = 0.283 * test_preds_ic50 + 0.283 * xgb_test_ic50 + 0.283 * lgb_test_ic50 + 0.151 * ridge_test_ic50
final_blend_cc50 = 0.283 * test_preds_cc50 + 0.283 * xgb_test_cc50 + 0.283 * lgb_test_cc50 + 0.151 * ridge_test_cc50
final_blend_si   = 0.283 * test_preds_si   + 0.283 * xgb_test_si   + 0.283 * lgb_test_si   + 0.151 * ridge_test_si

submission_fe_blend = pd.DataFrame({
    'index': test['index'],
    'IC50': final_blend_ic50,
    'CC50': final_blend_cc50,
    'SI': final_blend_si
})

submission_fe_blend.to_csv('submission_fe_blend.csv', index=False)
print(submission_fe_blend.head())

   index        IC50        CC50        SI
0      0  183.441552  320.244872  7.681156
1      1  267.374141  423.129036  6.346971
2      2  149.929545  464.081787  7.855419
3      3  292.294952  378.053063  5.893021
4      4  218.020110  358.400114  4.599111


In [65]:
feature_cols_custom_si = feature_cols + ['oof_IC50', 'oof_CC50']

global current_fold_cc50, current_fold_ic50

def physics_anchored_loss(y_true, y_pred, cc50_oof, ic50_oof, gamma=0.2):
    si_theoretical = cc50_oof / (ic50_oof + 1e-5)
    error_true = y_pred - y_true
    error_physics = y_pred - si_theoretical
    gradient = 2 * error_true + 2 * gamma * error_physics
    hessian = np.full_like(y_true, 2 + 2 * gamma)
    return gradient, hessian

def lgb_custom_objective(y_pred, dataset):
    y_true = dataset.get_label()
    global current_fold_cc50, current_fold_ic50
    grad, hess = physics_anchored_loss(y_true, y_pred, current_fold_cc50, current_fold_ic50, gamma=0.2)
    return grad, hess

custom_lgb_oof_si = np.zeros(len(train_final))
custom_lgb_test_si = np.zeros(len(test))

for fold in range(5):
    train_idx = train_final[train_final['fold'] != fold].index
    val_idx = train_final[train_final['fold'] == fold].index

    X_train = train_final.loc[train_idx, feature_cols_custom_si]
    y_train = train_final.loc[train_idx, 'SI']
    X_val = train_final.loc[val_idx, feature_cols_custom_si]
    y_val = train_final.loc[val_idx, 'SI']

    current_fold_cc50 = train_final.loc[train_idx, 'oof_CC50'].values
    current_fold_ic50 = train_final.loc[train_idx, 'oof_IC50'].values

    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

    params = {
        'objective': lgb_custom_objective,
        'metric': 'rmse',
        'learning_rate': 0.05,
        'max_depth': 6,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=1000,
        valid_sets=[lgb_val],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )

    custom_lgb_oof_si[val_idx] = model.predict(X_val)

    t_meta = test.copy()
    t_meta['oof_IC50'] = test_preds_ic50
    t_meta['oof_CC50'] = test_preds_cc50
    custom_lgb_test_si += model.predict(t_meta[feature_cols_custom_si]) / 5

In [66]:
final_physics_si = 0.283 * test_preds_si + 0.283 * xgb_test_si + 0.283 * custom_lgb_test_si + 0.151 * ridge_test_si

submission_physics_blend = pd.DataFrame({
    'index': test['index'],
    'IC50': final_blend_ic50,
    'CC50': final_blend_cc50,
    'SI': final_physics_si
})

submission_physics_blend.to_csv('submission_physics_blend.csv', index=False)
print(submission_physics_blend.head())

   index        IC50        CC50        SI
0      0  183.441552  320.244872  7.267385
1      1  267.374141  423.129036  5.830717
2      2  149.929545  464.081787  7.358683
3      3  292.294952  378.053063  5.522761
4      4  218.020110  358.400114  3.810945


In [67]:
train_raw = pd.read_csv('data/train.csv')
test_raw = pd.read_csv('data/test.csv')

target_cols = ['IC50, mM', 'CC50, mM', 'SI']
base_feature_cols = [col for col in train_raw.columns if col not in ["index"] + target_cols]

medians_raw = train_raw.groupby(base_feature_cols)[target_cols].transform('median')
train_raw[target_cols] = medians_raw
train_dirty = train_raw.drop_duplicates(subset=base_feature_cols, keep='first').reset_index(drop=True)

train_dirty = train_dirty.dropna(subset=target_cols).reset_index(drop=True)

def generate_fe_features(df):
    df_copy = df.copy()
    eps = 1e-5
    df_copy['custom_MolWt_per_Atom'] = df_copy['MolWt'] / (df_copy['HeavyAtomCount'] + eps)
    df_copy['custom_Polar_Ratio'] = df_copy['TPSA'] / (df_copy['LabuteASA'] + eps)
    df_copy['custom_Hydrophobic_Efficiency'] = df_copy['MolLogP'] / (df_copy['LabuteASA'] + eps)
    df_copy['custom_Electrons_per_Atom'] = df_copy['NumValenceElectrons'] / (df_copy['HeavyAtomCount'] + eps)
    return df_copy

train_dirty = generate_fe_features(train_dirty)
test_dirty = generate_fe_features(test_raw)


from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold(threshold=0.0)
selector.fit(train_dirty[base_feature_cols])
constant_f = [col for col, keep in zip(base_feature_cols, selector.get_support()) if not keep]

corr_matrix = train_dirty[base_feature_cols].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_f = [column for column in upper_tri.columns if any(upper_tri[column] > 0.95)]

final_fe_features = [col for col in base_feature_cols if col not in constant_f + high_corr_f]
final_fe_features = final_fe_features + ['custom_MolWt_per_Atom', 'custom_Polar_Ratio', 'custom_Hydrophobic_Efficiency', 'custom_Electrons_per_Atom']

print(f"Размерность датасета train_dirty (БЕЗ IQR): {train_dirty.shape}")
print(f"Количество признаков: {len(final_fe_features)}")

train_dirty['fold'] = -1
kf_dirty = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (train_idx, val_idx) in enumerate(kf_dirty.split(train_dirty)):
    train_dirty.loc[val_idx, 'fold'] = fold_idx

oof_cols = ['oof_IC50', 'oof_CC50', 'xgb_oof_IC50', 'xgb_oof_CC50', 'lgb_oof_IC50', 'lgb_oof_CC50', 'ridge_oof_IC50', 'ridge_oof_CC50']
for col in oof_cols:
    train_dirty[col] = 0.0

t_ic50, x_t_ic50, l_t_ic50, r_t_ic50 = np.zeros(len(test_dirty)), np.zeros(len(test_dirty)), np.zeros(len(test_dirty)), np.zeros(len(test_dirty))
t_cc50, x_t_cc50, l_t_cc50, r_t_cc50 = np.zeros(len(test_dirty)), np.zeros(len(test_dirty)), np.zeros(len(test_dirty)), np.zeros(len(test_dirty))

for fold in range(5):
    tr_idx, va_idx = train_dirty[train_dirty['fold'] != fold].index, train_dirty[train_dirty['fold'] == fold].index

    # CatBoost
    m_cb_ic = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0).fit(train_dirty.loc[tr_idx, final_fe_features], train_dirty.loc[tr_idx, 'IC50, mM'], eval_set=(train_dirty.loc[va_idx, final_fe_features], train_dirty.loc[va_idx, 'IC50, mM']), early_stopping_rounds=100, verbose=False)
    train_dirty.loc[va_idx, 'oof_IC50'] = m_cb_ic.predict(train_dirty.loc[va_idx, final_fe_features])
    t_ic50 += m_cb_ic.predict(test_dirty[final_fe_features]) / 5

    m_cb_cc = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0).fit(train_dirty.loc[tr_idx, final_fe_features], train_dirty.loc[tr_idx, 'CC50, mM'], eval_set=(train_dirty.loc[va_idx, final_fe_features], train_dirty.loc[va_idx, 'CC50, mM']), early_stopping_rounds=100, verbose=False)
    train_dirty.loc[va_idx, 'oof_CC50'] = m_cb_cc.predict(train_dirty.loc[va_idx, final_fe_features])
    t_cc50 += m_cb_cc.predict(test_dirty[final_fe_features]) / 5

    # XGBoost
    m_xgb_ic = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100).fit(train_dirty.loc[tr_idx, final_fe_features], train_dirty.loc[tr_idx, 'IC50, mM'], eval_set=[(train_dirty.loc[va_idx, final_fe_features], train_dirty.loc[va_idx, 'IC50, mM'])], verbose=False)
    train_dirty.loc[va_idx, 'xgb_oof_IC50'] = m_xgb_ic.predict(train_dirty.loc[va_idx, final_fe_features])
    x_t_ic50 += m_xgb_ic.predict(test_dirty[final_fe_features]) / 5

    m_xgb_cc = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100).fit(train_dirty.loc[tr_idx, final_fe_features], train_dirty.loc[tr_idx, 'CC50, mM'], eval_set=[(train_dirty.loc[va_idx, final_fe_features], train_dirty.loc[va_idx, 'CC50, mM'])], verbose=False)
    train_dirty.loc[va_idx, 'xgb_oof_CC50'] = m_xgb_cc.predict(train_dirty.loc[va_idx, final_fe_features])
    x_t_cc50 += m_xgb_cc.predict(test_dirty[final_fe_features]) / 5

    # LightGBM
    m_lgb_ic = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1).fit(train_dirty.loc[tr_idx, final_fe_features], train_dirty.loc[tr_idx, 'IC50, mM'], eval_set=[(train_dirty.loc[va_idx, final_fe_features], train_dirty.loc[va_idx, 'IC50, mM'])], callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
    train_dirty.loc[va_idx, 'lgb_oof_IC50'] = m_lgb_ic.predict(train_dirty.loc[va_idx, final_fe_features])
    l_t_ic50 += m_lgb_ic.predict(test_dirty[final_fe_features]) / 5

    m_lgb_cc = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1).fit(train_dirty.loc[tr_idx, final_fe_features], train_dirty.loc[tr_idx, 'CC50, mM'], eval_set=[(train_dirty.loc[va_idx, final_fe_features], train_dirty.loc[va_idx, 'CC50, mM'])], callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
    train_dirty.loc[va_idx, 'lgb_oof_CC50'] = m_lgb_cc.predict(train_dirty.loc[va_idx, final_fe_features])
    l_t_cc50 += m_lgb_cc.predict(test_dirty[final_fe_features]) / 5

    # Ridge
    imp, scl = SimpleImputer(strategy='median'), StandardScaler()
    X_tr_sc = scl.fit_transform(imp.fit_transform(train_dirty.loc[tr_idx, final_fe_features]))
    X_va_sc = scl.transform(imp.transform(train_dirty.loc[va_idx, final_fe_features]))
    t_sc = scl.transform(imp.transform(test_dirty[final_fe_features]))

    m_rg_ic = Ridge(alpha=10.0, random_state=42).fit(X_tr_sc, train_dirty.loc[tr_idx, 'IC50, mM'])
    train_dirty.loc[va_idx, 'ridge_oof_IC50'] = m_rg_ic.predict(X_va_sc)
    r_t_ic50 += m_rg_ic.predict(t_sc) / 5

    m_rg_cc = Ridge(alpha=10.0, random_state=42).fit(X_tr_sc, train_dirty.loc[tr_idx, 'CC50, mM'])
    train_dirty.loc[va_idx, 'ridge_oof_CC50'] = m_rg_cc.predict(X_va_sc)
    r_t_cc50 += m_rg_cc.predict(t_sc) / 5

t_si, x_t_si, l_t_si, r_t_si = np.zeros(len(test_dirty)), np.zeros(len(test_dirty)), np.zeros(len(test_dirty)), np.zeros(len(test_dirty))

for fold in range(5):
    tr_idx, va_idx = train_dirty[train_dirty['fold'] != fold].index, train_dirty[train_dirty['fold'] == fold].index

    f_cb, f_xgb, f_lgb, f_rg = final_fe_features + ['oof_IC50', 'oof_CC50'], final_fe_features + ['xgb_oof_IC50', 'xgb_oof_CC50'], final_fe_features + ['lgb_oof_IC50', 'lgb_oof_CC50'], final_fe_features + ['ridge_oof_IC50', 'ridge_oof_CC50']
    t_c, t_x, t_l, t_r = test_dirty.copy(), test_dirty.copy(), test_dirty.copy(), test_dirty.copy()
    t_c['oof_IC50'], t_c['oof_CC50'] = t_ic50, t_cc50
    t_x['xgb_oof_IC50'], t_x['xgb_oof_CC50'] = x_t_ic50, x_t_cc50
    t_l['lgb_oof_IC50'], t_lgb_col = l_t_ic50, l_t_cc50
    t_l['lgb_oof_CC50'] = l_t_cc50
    t_r['ridge_oof_IC50'], t_r['ridge_oof_CC50'] = r_t_ic50, r_t_cc50

    m_cb = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0).fit(train_dirty.loc[tr_idx, f_cb], train_dirty.loc[tr_idx, 'SI'], eval_set=(train_dirty.loc[va_idx, f_cb], train_dirty.loc[va_idx, 'SI']), early_stopping_rounds=100, verbose=False)
    t_si += m_cb.predict(t_c[f_cb]) / 5

    m_xgb = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, early_stopping_rounds=100).fit(train_dirty.loc[tr_idx, f_xgb], train_dirty.loc[tr_idx, 'SI'], eval_set=[(train_dirty.loc[va_idx, f_xgb], train_dirty.loc[va_idx, 'SI'])], verbose=False)
    x_t_si += m_xgb.predict(t_x[f_xgb]) / 5

    m_lgb = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbose=-1).fit(train_dirty.loc[tr_idx, f_lgb], train_dirty.loc[tr_idx, 'SI'], eval_set=[(train_dirty.loc[va_idx, f_lgb], train_dirty.loc[va_idx, 'SI'])], callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
    l_t_si += m_lgb.predict(t_l[f_lgb]) / 5

    X_tr_sc = scl.fit_transform(imp.fit_transform(train_dirty.loc[tr_idx, f_rg]))
    r_t_si += Ridge(alpha=10.0, random_state=42).fit(X_tr_sc, train_dirty.loc[tr_idx, 'SI']).predict(scl.transform(imp.transform(t_r[f_rg]))) / 5

f_b_ic50 = 0.283 * t_ic50 + 0.283 * x_t_ic50 + 0.283 * l_t_ic50 + 0.151 * r_t_ic50
f_b_cc50 = 0.283 * t_cc50 + 0.283 * x_t_cc50 + 0.283 * l_t_cc50 + 0.151 * r_t_cc50
f_b_si   = 0.283 * t_si   + 0.283 * x_t_si   + 0.283 * l_t_si   + 0.151 * r_t_si
submission_no_iqr = pd.DataFrame({
    'index': test_raw['index'],
    'IC50': f_b_ic50,
    'CC50': f_b_cc50,
    'SI': f_b_si
    })
submission_no_iqr.to_csv('submission_no_iqr.csv', index=False)
print("Файл эксперимета submission_no_iqr.csv успешно создан!")
print(submission_no_iqr.head())

Размерность датасета train_dirty (БЕЗ IQR): (628, 218)
Количество признаков: 162
Файл эксперимета submission_no_iqr.csv успешно создан!
   index        IC50        CC50         SI
0      0  175.437555  364.585032  15.579480
1      1  241.892929  425.368667  14.534453
2      2  122.782062  453.854603  17.237507
3      3  258.458508  377.097128   6.327018
4      4  207.087039  318.073443   4.993954


In [68]:

feature_cols_custom_si = final_fe_features + ['oof_IC50', 'oof_CC50']

global current_fold_cc50, current_fold_ic50

def physics_anchored_loss(y_true, y_pred, cc50_oof, ic50_oof, gamma=0.2):
    si_theoretical = cc50_oof / (ic50_oof + 1e-5)
    error_true = y_pred - y_true
    error_physics = y_pred - si_theoretical
    gradient = 2 * error_true + 2 * gamma * error_physics
    hessian = np.full_like(y_true, 2 + 2 * gamma)
    return gradient, hessian

def lgb_custom_objective(y_pred, dataset):
    y_true = dataset.get_label()
    global current_fold_cc50, current_fold_ic50
    grad, hess = physics_anchored_loss(y_true, y_pred, current_fold_cc50, current_fold_ic50, gamma=0.2)
    return grad, hess

custom_lgb_test_si = np.zeros(len(test_dirty))

for fold in range(5):
    tr_idx, va_idx = train_dirty[train_dirty['fold'] != fold].index, train_dirty[train_dirty['fold'] == fold].index

    X_train = train_dirty.loc[tr_idx, feature_cols_custom_si]
    y_train = train_dirty.loc[tr_idx, 'SI']
    X_val = train_dirty.loc[va_idx, feature_cols_custom_si]

    current_fold_cc50 = train_dirty.loc[tr_idx, 'oof_CC50'].values
    current_fold_ic50 = train_dirty.loc[tr_idx, 'oof_IC50'].values

    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_val = lgb.Dataset(X_val, label=train_dirty.loc[va_idx, 'SI'], reference=lgb_train)

    params = {
        'objective': lgb_custom_objective,
        'metric': 'rmse',
        'learning_rate': 0.05,
        'max_depth': 6,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

    model = lgb.train(params, lgb_train, num_boost_round=1000, valid_sets=[lgb_val], callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])

    t_meta = test_dirty.copy()
    t_meta['oof_IC50'] = t_ic50
    t_meta['oof_CC50'] = t_cc50
    custom_lgb_test_si += model.predict(t_meta[feature_cols_custom_si]) / 5


final_dirty_si = 0.283 * t_si + 0.283 * x_t_si + 0.283 * custom_lgb_test_si + 0.151 * r_t_si

submission_physics_dirty_blend = pd.DataFrame({
    'index': test_raw['index'],
    'IC50': f_b_ic50,
    'CC50': f_b_cc50,
    'SI': final_dirty_si
})

submission_physics_dirty_blend.to_csv('submission_physics_dirty_blend.csv', index=False)
print("Файл submission_physics_dirty_blend.csv создан")
print(submission_physics_dirty_blend.head())


Файл submission_physics_dirty_blend.csv создан
   index        IC50        CC50         SI
0      0  175.437555  364.585032  13.413419
1      1  241.892929  425.368667  13.499705
2      2  122.782062  453.854603  15.086403
3      3  258.458508  377.097128   3.754195
4      4  207.087039  318.073443   2.997796
